In [1]:
# Cell 1: Environment setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
# Cell 2: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

In [3]:
# Cell 3: Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.12.0.dev20260312+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [4]:
# Cell 4: Load ASVspoof data
# Paths
ASVSPOOF_ROOT = r"C:\deepfake-project\data\asvspoof"
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")
 
# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
# Convert labels to integers — 0 = bonafide, 1 = spoof
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})
 
# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))
 
print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

Training samples:   25380
Dev samples:        24844
Train label split:  {1: 22800, 0: 2580}
Dev label split:    {1: 22296, 0: 2548}


In [ ]:
# Cell 5: Dataset with WhatsApp-style augmentation
import os
import tempfile
import subprocess

SAMPLE_RATE = 16000
MAX_SAMPLES = 64000  # 4 seconds
 
class ASVspoofDataset(Dataset):
    """
    Dataset with data augmentation to simulate real-world conditions:
    - Codec compression (WhatsApp uses Opus at 16-32kbps)
    - Background noise
    - Volume variations
    """
    
    def __init__(self, df, feature_extractor, augment=True):
        self.df = df
        self.feature_extractor = feature_extractor
        self.augment = augment
    
    def __len__(self):
        return len(self.df)
    
    def apply_opus_compression(self, audio_tensor, sr=16000):
        """Apply real Opus codec compression via FFmpeg"""
        # Convert tensor to numpy
        audio_np = audio_tensor.squeeze(0).numpy()
        
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_in, \
             tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as tmp_ogg, \
             tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_out:
            
            try:
                # Write original audio
                sf.write(tmp_in.name, audio_np, sr)
                
                # Encode to Opus (WhatsApp typically uses 16-32kbps)
                bitrate = random.choice([16, 24, 32])  # kbps
                result = subprocess.run([
                    "ffmpeg", "-y", "-i", tmp_in.name,
                    "-c:a", "libopus",
                    "-b:a", f"{bitrate}k",
                    "-ar", str(sr),
                    tmp_ogg.name
                ], capture_output=True, text=True)
                
                if result.returncode != 0:
                    # Fallback to original audio if FFmpeg fails
                    print(f"Warning: Opus encoding failed, using original audio")
                    return audio_tensor
                
                # Decode back to WAV
                result = subprocess.run([
                    "ffmpeg", "-y", "-i", tmp_ogg.name,
                    "-ar", str(sr),
                    "-ac", "1",
                    tmp_out.name
                ], capture_output=True, text=True)
                
                if result.returncode != 0:
                    print(f"Warning: Opus decoding failed, using original audio")
                    return audio_tensor
                
                # Read compressed audio
                compressed_audio, _ = sf.read(tmp_out.name, dtype="float32")
                
                return torch.from_numpy(compressed_audio).unsqueeze(0)
                
            finally:
                # Cleanup temp files
                for tmp_file in [tmp_in.name, tmp_ogg.name, tmp_out.name]:
                    if os.path.exists(tmp_file):
                        try:
                            os.unlink(tmp_file)
                        except:
                            pass
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load the audio file
        audio, sr = sf.read(row["path"])
        audio = audio.astype(np.float32)
        
        # Convert to tensor for augmentation
        audio_tensor = torch.from_numpy(audio).unsqueeze(0)
        
        # === DATA AUGMENTATION (only for training) ===
        if self.augment and random.random() < 0.7:  # 70% of training samples
            
            # 1. Simulate codec compression (WhatsApp uses Opus)
            if random.random() < 0.5:
                audio_tensor = self.apply_opus_compression(audio_tensor, SAMPLE_RATE)
            
            # 2. Add background noise (microphone noise, room tone)
            if random.random() < 0.4:
                noise_level = random.uniform(0.001, 0.01)
                noise = torch.randn_like(audio_tensor) * noise_level
                audio_tensor = audio_tensor + noise
            
            # 3. Volume variation (people speak at different distances)
            if random.random() < 0.3:
                volume_factor = random.uniform(0.7, 1.3)
                audio_tensor = audio_tensor * volume_factor
            
            # 4. Slight pitch shift (simulates different devices)
            if random.random() < 0.2:
                shift_factor = random.uniform(0.95, 1.05)
                target_length = int(audio_tensor.shape[1] * shift_factor)
                audio_tensor = torchaudio.functional.resample(
                    audio_tensor, 
                    SAMPLE_RATE, 
                    int(SAMPLE_RATE * shift_factor)
                )
                # Resample back to original rate
                audio_tensor = torchaudio.functional.resample(
                    audio_tensor,
                    int(SAMPLE_RATE * shift_factor),
                    SAMPLE_RATE
                )
        
        # Convert back to numpy
        audio = audio_tensor.squeeze(0).numpy()
        
        # Pad or truncate to exactly 4 seconds
        if len(audio) >= MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
        
        # Normalise amplitude
        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak
        
        # Apply wav2vec2 feature extraction
        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )
        
        return {
            "input_values": inputs["input_values"].squeeze(0).to(torch.bfloat16),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [6]:
# Cell 6: Load model with proper label mapping
MODEL_NAME = "facebook/wav2vec2-base"
device     = torch.device("cuda")
 
print("Loading feature extractor...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
 
print("Loading model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
 
# Set proper label names (this will be saved with the model)
model.config.id2label = {0: "bonafide", 1: "spoof"}
model.config.label2id = {"bonafide": 0, "spoof": 1}
 
# Freeze the CNN feature extractor
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False
 
# Freeze the bottom 6 transformer layers (half of 12)
for i in range(6):
    for param in model.wav2vec2.encoder.layers[i].parameters():
        param.requires_grad = False
 
# Move model to GPU then convert weights to bfloat16
model = model.to(device)
model = model.to(torch.bfloat16)
 
# Allow TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
 
print(f"Model loaded on {device}")
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Frozen parameters:    {total - trainable:,} / {total:,}")
print(f"\nLabel mapping: {model.config.id2label}")

Loading feature extractor...


Loading model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
Trainable parameters: 47,841,410 / 94,569,090
Frozen parameters:    46,727,680 / 94,569,090

Label mapping: {0: 'bonafide', 1: 'spoof'}


In [7]:
# Cell 7: Create data loaders
from torch.utils.data import WeightedRandomSampler
 
# Create dataset objects with augmentation
train_dataset = ASVspoofDataset(train_df, feature_extractor, augment=True)
dev_dataset   = ASVspoofDataset(dev_df,   feature_extractor, augment=False)  # No augmentation for validation
 
# Handle class imbalance using a weighted sampler
class_counts  = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map({0: class_weights[0], 1: class_weights[1]}).values
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
 
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler,
    num_workers=0
)
 
dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)
 
print(f"Training batches:   {len(train_loader)}")
print(f"Development batches: {len(dev_loader)}")
print("✓ Data augmentation enabled for training set")

Training batches:   3173
Development batches: 3106
✓ Data augmentation enabled for training set


c:\Users\tkell\anaconda3\envs\deepfake\Lib\site-packages\torch\utils\data\sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


In [8]:
# Cell 8: Optimizer and evaluation function
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score
 
# AdamW optimiser
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)
 
# Linear warmup scheduler
total_steps  = len(train_loader) * 5  # 5 epochs instead of 3
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
 
def evaluate(model, loader, device):
    """Run model on dev set and return loss, accuracy and ROC-AUC."""
    model.eval()
    total_loss, correct, all_labels, all_probs = 0, 0, [], []
 
    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            labels       = batch["label"].to(device)
 
            outputs = model(input_values=input_values, labels=labels)
            total_loss += outputs.loss.item()
 
            probs  = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()
 
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, roc_auc
 
print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

Optimiser and evaluation function ready
Total training steps: 15865
Warmup steps:         1586


In [9]:
# Cell 9: Training loop
from torch.amp import autocast
from sklearn.metrics import roc_auc_score
 
EPOCHS       = 5  # Increased from 3
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\wav2vec2_finetuned"
best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 4  # Increased patience
 
os.makedirs(SAVE_DIR, exist_ok=True)
 
print("Starting training with data augmentation...")
print(f"Evaluating every {EVAL_STEPS} steps — best model saved to {SAVE_DIR}")
print("-" * 60)
 
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    step       = 0
 
    for batch in train_loader:
        input_values = batch["input_values"].to(device)
        labels       = batch["label"].to(device)
 
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()
 
        epoch_loss += loss.item()
        step       += 1
 
        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")
 
        if step % EVAL_STEPS == 0:
            dev_loss, dev_acc, dev_roc = evaluate(model, dev_loader, device)
            print(f"\n>>> Eval @ step {step} | Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")
 
            if dev_roc > best_roc_auc:
                best_roc_auc = dev_roc
                patience     = 0
                model.save_pretrained(SAVE_DIR)
                feature_extractor.save_pretrained(SAVE_DIR)
                print(f"    ✓ New best model saved (ROC-AUC: {best_roc_auc:.4f})")
            else:
                patience += 1
                print(f"    No improvement — patience {patience}/{PATIENCE_MAX}")
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break
 
            model.train()
 
    if patience >= PATIENCE_MAX:
        break
 
    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")
 
print("-" * 60)
print(f"Training complete | Best ROC-AUC: {best_roc_auc:.4f}")
print(f"Best model saved to: {SAVE_DIR}")

Starting training with data augmentation...
Evaluating every 200 steps — best model saved to C:\deepfake-project\models\wav2vec2_finetuned
------------------------------------------------------------
Epoch 1 | Step 50/3173 | Loss: 0.6940
Epoch 1 | Step 100/3173 | Loss: 0.6934
Epoch 1 | Step 150/3173 | Loss: 0.6926
Epoch 1 | Step 200/3173 | Loss: 0.6911

>>> Eval @ step 200 | Loss: 0.7118 | Acc: 0.1327 | ROC-AUC: 0.8665


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.8665)
Epoch 1 | Step 250/3173 | Loss: 0.6877
Epoch 1 | Step 300/3173 | Loss: 0.6710
Epoch 1 | Step 350/3173 | Loss: 0.6360
Epoch 1 | Step 400/3173 | Loss: 0.5987

>>> Eval @ step 400 | Loss: 0.5462 | Acc: 0.7833 | ROC-AUC: 0.9804


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9804)
Epoch 1 | Step 450/3173 | Loss: 0.5596
Epoch 1 | Step 500/3173 | Loss: 0.5215
Epoch 1 | Step 550/3173 | Loss: 0.4869
Epoch 1 | Step 600/3173 | Loss: 0.4566

>>> Eval @ step 600 | Loss: 0.5462 | Acc: 0.8377 | ROC-AUC: 0.9909


...

    ✓ New best model saved (ROC-AUC: 0.9984)
Epoch 1 | Step 1850/3173 | Loss: 0.1881
Epoch 1 | Step 1900/3173 | Loss: 0.1838
Epoch 1 | Step 1950/3173 | Loss: 0.1804
Epoch 1 | Step 2000/3173 | Loss: 0.1768

>>> Eval @ step 2000 | Loss: 0.1437 | Acc: 0.9713 | ROC-AUC: 0.9993


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9993)
Epoch 1 | Step 2050/3173 | Loss: 0.1729
Epoch 1 | Step 2100/3173 | Loss: 0.1697
Epoch 1 | Step 2150/3173 | Loss: 0.1670
Epoch 1 | Step 2200/3173 | Loss: 0.1634

>>> Eval @ step 2200 | Loss: 0.2921 | Acc: 0.9443 | ROC-AUC: 0.9990
    No improvement — patience 1/4
Epoch 1 | Step 2250/3173 | Loss: 0.1599
Epoch 1 | Step 2300/3173 | Loss: 0.1569
Epoch 1 | Step 2350/3173 | Loss: 0.1547
Epoch 1 | Step 2400/3173 | Loss: 0.1522

>>> Eval @ step 2400 | Loss: 0.1268 | Acc: 0.9735 | ROC-AUC: 0.9996


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9996)
Epoch 1 | Step 2450/3173 | Loss: 0.1496
Epoch 1 | Step 2500/3173 | Loss: 0.1472
Epoch 1 | Step 2550/3173 | Loss: 0.1451
Epoch 1 | Step 2600/3173 | Loss: 0.1423

>>> Eval @ step 2600 | Loss: 0.1319 | Acc: 0.9734 | ROC-AUC: 0.9996
    No improvement — patience 1/4
Epoch 1 | Step 2650/3173 | Loss: 0.1403
Epoch 1 | Step 2700/3173 | Loss: 0.1394
Epoch 1 | Step 2750/3173 | Loss: 0.1369
Epoch 1 | Step 2800/3173 | Loss: 0.1348

>>> Eval @ step 2800 | Loss: 0.1692 | Acc: 0.9661 | ROC-AUC: 0.9995
    No improvement — patience 2/4
Epoch 1 | Step 2850/3173 | Loss: 0.1327
Epoch 1 | Step 2900/3173 | Loss: 0.1305
Epoch 1 | Step 2950/3173 | Loss: 0.1289
Epoch 1 | Step 3000/3173 | Loss: 0.1274

>>> Eval @ step 3000 | Loss: 0.0881 | Acc: 0.9818 | ROC-AUC: 0.9995
    No improvement — patience 3/4
Epoch 1 | Step 3050/3173 | Loss: 0.1258
Epoch 1 | Step 3100/3173 | Loss: 0.1239
Epoch 1 | Step 3150/3173 | Loss: 0.1224

Epoch 1 complete | Avg loss: 0.1215

Epoch 2 

In [10]:
# Cell 10: Test saved model
saved_model = Wav2Vec2ForSequenceClassification.from_pretrained(SAVE_DIR)
saved_extractor = Wav2Vec2FeatureExtractor.from_pretrained(SAVE_DIR)
 
saved_model = saved_model.to(device)
saved_model = saved_model.to(torch.bfloat16)
saved_model.eval()
 
print(f"\n=== Saved Model Configuration ===")
print(f"id2label: {saved_model.config.id2label}")
print(f"label2id: {saved_model.config.label2id}")
 
def predict(audio_path):
    audio, sr = sf.read(audio_path)
    audio = audio.astype(np.float32)
 
    if len(audio) >= MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    else:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
 
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
 
    inputs = saved_extractor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=False
    )
 
    input_values = inputs["input_values"].to(device).to(torch.bfloat16)
 
    with torch.no_grad():
        outputs = saved_model(input_values=input_values)
        probs = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
 
    return {
        "bonafide": round(probs[0][0].item(), 4),
        "spoof":    round(probs[0][1].item(), 4),
        "verdict":  "REAL" if probs[0][0] > probs[0][1] else "FAKE"
    }
 
# Test on dev set
real_file = dev_df[dev_df["label"] == 0].iloc[0]["path"]
fake_file = dev_df[dev_df["label"] == 1].iloc[0]["path"]
 
print("\n=== Testing saved model ===")
print(f"\nReal audio: {os.path.basename(real_file)}")
print(predict(real_file))
 
print(f"\nFake audio: {os.path.basename(fake_file)}")
print(predict(fake_file))

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


=== Saved Model Configuration ===
id2label: {0: 'bonafide', 1: 'spoof'}
label2id: {'bonafide': 0, 'spoof': 1}

=== Testing saved model ===

Real audio: LA_D_1047731.flac
{'bonafide': 1.0, 'spoof': 0.0012, 'verdict': 'REAL'}

Fake audio: LA_D_1008730.flac
{'bonafide': 0.0017, 'spoof': 1.0, 'verdict': 'FAKE'}
